# Notebook 01 — Page schemas, frontmatter, and wiki-level config

**Purpose:** Establish the data layer. Zero API calls. Everything downstream depends on these models.

In [ ]:
# Enable autoreload so edits to engine modules are picked up automatically.
%load_ext autoreload
%autoreload 2

In [ ]:
# Import shared notebook dependencies for schema validation and frontmatter parsing.
from pathlib import Path
from datetime import date
from enum import Enum
from typing import Literal, Annotated

from pydantic import BaseModel, Field, ValidationError, field_validator
import frontmatter

In [ ]:
# Load page schema models from the engine package.
import engine.models.pages as page_models

In [ ]:
# Derive and print the §6.2 per-page contract table directly from model definitions.
from typing import get_args


def allows_none(field) -> bool:
    return type(None) in get_args(field.annotation)


def min_length_of(field):
    for meta in field.metadata:
        value = getattr(meta, "min_length", None)
        if value is not None:
            return value
    return None


def tightenings_for(subclass, base_class=page_models.BasePage) -> list[str]:
    base_fields = base_class.model_fields
    sub_fields = subclass.model_fields
    tightenings: list[str] = []

    for name, sub_field in sub_fields.items():
        if name not in base_fields:
            continue

        base_field = base_fields[name]

        if name == "type":
            tightenings.append(f"type pinned to '{sub_field.default.value}'")
            continue

        if (not base_field.is_required()) and sub_field.is_required():
            tightenings.append(f"{name}: optional -> required")

        if allows_none(base_field) and (not allows_none(sub_field)):
            tightenings.append(f"{name}: nullable -> non-null")

        base_min_len = min_length_of(base_field)
        sub_min_len = min_length_of(sub_field)
        if sub_min_len is not None and (base_min_len is None or sub_min_len > base_min_len):
            tightenings.append(f"{name}: min_length {base_min_len or 0} -> {sub_min_len}")

    return tightenings


subclass_by_type = {
    cls.model_fields["type"].default: cls
    for cls in page_models.BasePage.__subclasses__()
    if "type" in cls.model_fields
}

print("§6.2 Contract Table (derived from code)")
print("=" * 44)
for page_type in page_models.PageType:
    cls = subclass_by_type.get(page_type)
    if cls is None:
        print(f"\n- {page_type.value}: <missing model class>")
        continue

    tight = tightenings_for(cls)
    print(f"\n- {page_type.value} ({cls.__name__})")
    if tight:
        for item in tight:
            print(f"  - {item}")
    else:
        print("  - no field tightenings")

In [ ]:
# Parse, validate, mutate, and round-trip a valid sample page through frontmatter.
#
# Goal: prove that parse -> validate -> mutate -> serialize -> re-parse -> re-validate
# preserves the page model semantically. Equality is asserted on the re-validated
# Pydantic object (not on raw metadata dicts) because YAML serialization can normalize
# representation (e.g. None <-> null, date objects <-> ISO strings) without changing
# meaning. Body markdown is checked separately as a string.
from pydantic import TypeAdapter

from engine.models.pages import Page

text = """---
title: Apollo Growth Team
type: entity
status: active
created: 2026-04-25
last_synced: 2026-04-30
confidence: high
sources: []
owners:
  - "[[teams/apollo-growth]]"
tags:
  - team
  - growth
related:
  - "[[concepts/north-star-metric]]"
supersedes:
contradicts: []
validation_errors: []
---

# Apollo Growth Team

The Apollo Growth Team owns acquisition experiments for the self-serve funnel.

## Scope

- Runs weekly experiment planning and review
- Maintains activation metrics and dashboards
- Partners with Product and Data on instrumentation

## Current Priorities

1. Improve signup-to-activation conversion
2. Reduce time-to-first-value
3. Standardize experiment readouts
"""

post = frontmatter.loads(text)
page = TypeAdapter(Page).validate_python(post.metadata)
page2 = page.model_copy(update={"title": "New title"})
serialized = frontmatter.dumps(
    frontmatter.Post(content=post.content, **page2.model_dump(mode="json"))
)
re_parsed = frontmatter.loads(serialized)
re_validated = TypeAdapter(Page).validate_python(re_parsed.metadata)

assert re_validated == page2, "round-tripped page does not equal the mutated original"
assert re_parsed.content.strip() == post.content.strip(), "body markdown was not preserved"
assert re_validated.title == "New title", "title mutation did not survive the round trip"
print("round-trip ok:", re_validated.title, "/", re_validated.type.value)


In [ ]:
# Export JSON schemas for all page types to _ops/schemas.
from engine.models.pages import EntityPage, ConceptPage, SourcePage, DecisionPage, MeetingPage, MetricPage, QAPage, AnalysisPage      # all 8
import json
out = Path("_ops/schemas")
out.mkdir(parents=True, exist_ok=True)
for cls in [EntityPage, ConceptPage, SourcePage, DecisionPage, MeetingPage, MetricPage, QAPage, AnalysisPage]:
    (out / f"{cls.model_fields['type'].default.value}.json").write_text(
        json.dumps(cls.model_json_schema(), indent=2)
    )

In [ ]:
# Exercise intentional validation failures to inspect error quality for retry loops.
from pydantic import TypeAdapter

from engine.models.pages import DecisionPage, EntityPage, Page


def run_case(name: str, fn) -> None:
    print(f"\n=== {name} ===")
    try:
        fn()
        print("UNEXPECTED: validation passed")
    except ValidationError as exc:
        print(exc)


base_entity = {
    "title": "Apollo Growth Team",
    "type": "entity",
    "status": "active",
    "created": "2026-04-25",
    "last_synced": "2026-04-30",
    "sources": [],
    "owners": ["[[teams/apollo-growth]]"],
    "tags": ["team"],
    "related": ["[[concepts/north-star-metric]]"],
    "supersedes": None,
    "contradicts": [],
    "validation_errors": [],
}

run_case(
    "EntityPage with empty tags=[]",
    lambda: EntityPage.model_validate({**base_entity, "tags": []}),
)

run_case(
    "DecisionPage missing confidence",
    lambda: DecisionPage.model_validate(
        {
            **base_entity,
            "type": "decision",
            "owners": ["[[teams/apollo-growth]]"],
        }
    ),
)

run_case(
    "Archived page with no archived_date",
    lambda: EntityPage.model_validate(
        {
            **base_entity,
            "status": "archived",
            "archived_reason": "manual",
            "archived_date": None,
        }
    ),
)

run_case(
    "related: ['Apollo'] bare name",
    lambda: EntityPage.model_validate({**base_entity, "related": ["Apollo"]}),
)

run_case(
    "type: 'rumor' not in enum (discriminated union)",
    lambda: TypeAdapter(Page).validate_python({**base_entity, "type": "rumor"}),
)

In [ ]:
# Render a valid sample page using rich for frontmatter + body preview.
try:
    from rich.console import Console
    from rich.syntax import Syntax
except ImportError:
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "rich"])
    from rich.console import Console
    from rich.syntax import Syntax

if "post" not in globals():
    post = frontmatter.loads(text)

console = Console()
console.print(Syntax(frontmatter.dumps(post), "yaml"))
console.print(post.content)

## Wiki-level config contract (§6.4)

`MarginaliaConfig` bridges filesystem state (`purpose.md` and `AGENTS.md`) into agent runtime context.

- Agents receive `purpose_body` and `agents_body` verbatim as system-prompt material.
- `in_scope` and `out_of_scope` are convenience extraction for CLI display and operator ergonomics.
- The extracted bullet lists are not authoritative; the raw markdown bodies are.

In [ ]:
# Create wiki-level config files and demo MarginaliaConfig.load().
from engine.models.wiki_config import MarginaliaConfig

path = Path("data/poc-wiki")

wiki_root = Path("data/poc-wiki")
wiki_root.mkdir(parents=True, exist_ok=True)

purpose_text = """# Purpose

This wiki captures durable operational knowledge for the Marginalia engine.

## In scope

- Engine architecture decisions and trade-offs
- Agent contracts, prompts, and model routing policy
- Ingest, synthesis, QA, and lint operational playbooks
- Canonical runbooks and incident learnings

## Out of scope

- Personal notes unrelated to repository operation
- One-off scratchpad experiments with no long-term value
- Secrets, credentials, or production tokens
"""

agents_text = """# AGENTS

## Terminology rules

- We say customer, not client.
- We use source, not document, for upstream artifacts.
- We use wiki page, not note.

## Formatting rules

- All dates are ISO-8601 (`YYYY-MM-DD`).
- Prefer explicit wikilinks: `[[namespace/path]]`.
- Keep status terms aligned with enums: draft, active, stale, archived, superseded.
"""

(wiki_root / "purpose.md").write_text(purpose_text, encoding="utf-8")
(wiki_root / "AGENTS.md").write_text(agents_text, encoding="utf-8")

cfg = MarginaliaConfig.load("data/poc-wiki")
print(cfg.in_scope, cfg.out_of_scope)
print(len(cfg.purpose_body), len(cfg.agents_body))

In [ ]:
# Negative test: confirm missing wiki config files raise a loud WikiConfigError.
from engine.models.wiki_config import MarginaliaConfig, WikiConfigError

try:
    MarginaliaConfig.load("/some/empty/dir")
    print("UNEXPECTED: load succeeded")
except WikiConfigError as exc:
    print(exc)

In [ ]:
# What to extract
#
# This notebook proves the page-schema and wiki-config contracts end-to-end,
# including validation behavior, schema export, and config loading.
#
# Artifacts that landed in engine/:
# - engine/models/pages.py ✓
# - engine/models/wiki_config.py ✓
# - engine/models/__init__.py ✓
# - _ops/schemas/*.json (created in step 2 above)